In [ ]:
import os
import sys
import time
from IPython.display import Markdown, display, Image
from dotenv import load_dotenv

# --- DYNAMIC PATH RESOLUTION ---
current_dir = os.getcwd()
if current_dir.endswith('notebooks'):
    project_root = os.path.dirname(current_dir)
else:
    project_root = current_dir

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root set to: {project_root}")

# Load environment variables
load_dotenv()

# Import the Variant 2 Baseline Graph
from workflow_engine.orchestrators.baseline_graph import build_baseline_graph

In [ ]:
# Setup Test Variables
# Make sure to point this to a real CSV file in your data/raw folder
raw_path = os.path.join(project_root, "data", "raw", "paysim_100k_sample.csv").replace('\\', '/')
target_col = "isFraud" # Update to match your CSV (target, isFraud, Churn)

initial_state = {
    "messages": ["Starting Notebook baseline benchmark."],
    "user_request": f"Clean the data, engineer features, and train a classification model to predict '{target_col}'.",
    "target_variable": target_col,
    "raw_dataset_path": raw_path,
    "current_dataset_path": raw_path,
    "artifacts": {},
    "current_step": "start",
    "error_flag": False,
    "error_message": "",
    "revision_count": 0,
    "user_preferences": {},
    "total_sleep_time": 0.0,
    "next_node": ""
}

# Build the Graph
app = build_baseline_graph()

print("🚀 Starting Variant 2 Baseline Execution...")
print("⏱️  Clock is running. Please wait...")

# --- PURE EXECUTION TIMING ---
start_time = time.time()

try:
    # Using .invoke() blocks the thread and runs the entire pipeline silently
    final_state = app.invoke(initial_state)
except Exception as e:
    print(f"❌ Graph execution crashed: {e}")
    final_state = {"error_flag": True, "error_message": str(e)}

end_time = time.time()
gross_execution_time = end_time - start_time
total_sleep = final_state.get("total_sleep_time", 0.0)
pure_agent_execution_time = gross_execution_time - total_sleep

In [ ]:
if not final_state.get("error_flag"):
    print("\n" + "="*60)
    print("🚀 PIPELINE EXECUTION SUCCESSFUL!")
    print("="*60)
    print(f"⏱️ Gross Wall-Clock Time: {gross_execution_time:.2f} seconds")
    print(f"💤 Total Artificial Sleep: {total_sleep:.2f} seconds")
    print(f"⚡ PURE AGENT EXECUTION TIME: {pure_agent_execution_time:.2f} seconds")
    
    artifacts = final_state.get("artifacts", {})
    
    # Render the Confusion Matrix
    if "confusion_matrix" in artifacts and os.path.exists(artifacts["confusion_matrix"]):
        print("\n--- Model Evaluation ---")
        display(Image(filename=artifacts["confusion_matrix"]))
    
    # Render the Final Markdown Report
    if "final_report" in artifacts and os.path.exists(artifacts["final_report"]):
        print("\n--- Final Generated Report ---")
        with open(artifacts["final_report"], "r", encoding="utf-8") as f:
            display(Markdown(f.read()))
    else:
        print("No final report found in artifacts.")
else:
    print("\n⚠️ Pipeline failed to complete successfully. Check the error logs.")
    print(f"Error Message: {final_state.get('error_message')}")
    print(f"⏱️ Time to failure: {gross_execution_time:.2f} seconds")